In [1]:
from google.colab import files

# UPLOAD_ZIP
uploaded = files.upload()

Saving mini_corpus.zip to mini_corpus.zip


In [2]:
import zipfile
import os

zip_file = "mini_corpus.zip"
extract_dir = "mini_corpus"

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("EXTRACT")

EXTRACT


In [3]:
!pip install -q git+https://github.com/openai/whisper.git
!pip install -q torchaudio jiwer

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 45.3 MB/s eta

In [4]:
import whisper
import os
import pandas as pd
import time
import jiwer

# LODING_MODEL
model = whisper.load_model("large")

# ROAD_FILE
audio_dir = "mini_corpus/clips"
tsv_path = "mini_corpus/validated.tsv"

# خواندن tsv برای مقایسه با ground truth
df = pd.read_csv(tsv_path, sep="\t")

# RESULTS
results = []

# TEST_100_SAMPLE
for idx, row in df.iterrows():
    file_name = row["path"]
    true_text = str(row["sentence"]).strip()
    file_path = os.path.join(audio_dir, file_name)

    if not os.path.exists(file_path):
        continue

    start = time.time()
    try:
        output = model.transcribe(file_path, fp16=True,language='fa',)
    except Exception as e:
        print(f"ERROR{file_name}: {e}")
        continue
    end = time.time()

    predicted_text = output["text"].strip()
    elapsed = end - start
    wer = jiwer.wer(true_text.lower(), predicted_text.lower())

    results.append({
        "filename": file_name,
        "true_text": true_text,
        "predicted_text": predicted_text,
        "time_sec": round(elapsed, 2),
        "WER": round(wer, 4)
    })

print("TEST")

100%|█████████████████████████████████████| 2.88G/2.88G [00:57<00:00, 53.4MiB/s]


TEST


In [5]:
results_df = pd.DataFrame(results)
print(results_df.head(10))  # نمایش 10 مورد اول

avg_time = results_df["time_sec"].mean()
avg_wer = results_df["WER"].mean()

print(f"\n میانگین زمان پردازش: {avg_time:.2f} ثانیه")
print(f" میانگین خطای WER: {avg_wer:.2%}")

                       filename                  true_text  \
0  common_voice_fa_18325365.mp3     از مهمونداری کنار بکشم   
1  common_voice_fa_18960256.mp3      خب ، تو چیكار می كنی؟   
2  common_voice_fa_33143153.mp3              اتوبوس مسافری   
3  common_voice_fa_19446941.mp3              آه، نه اصلاُ!   
4  common_voice_fa_18557643.mp3     دو استایل متفاوت دارین   
5  common_voice_fa_21653951.mp3  ما تحقیقات انجام می دادیم   
6  common_voice_fa_19209861.mp3     دو روز قبل از کریسمس ؟   
7  common_voice_fa_19887398.mp3        ساعت های کاری چیست؟   
8  common_voice_fa_35176695.mp3                     مسابقه   
9  common_voice_fa_19208010.mp3     اعصابم اون شب خورد بود   

              predicted_text  time_sec     WER  
0     از مهمانداری کنار بکشم      3.93  0.2500  
1            خوبتو چیکارن کن      0.81  1.0000  
2             اوتوبوس مسافری      0.67  0.5000  
3                    نه اصلا      0.60  0.6667  
4       تو بسه لمت فاوت داری      0.90  1.2500  
5  ما تحقیقات انجام می 

In [6]:
results_df.to_csv("whisper_test_results.csv", index=False)
from google.colab import files
files.download("whisper_test_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>